# Demographics of sets

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
w = pd.read_csv('windows_matched.csv')
c = pd.read_csv('cohort_matched.csv')
s = pd.read_csv('split_index_matched.csv')

In [ ]:
s.columns = s.columns.str.lower()
c.columns = c.columns.str.lower()

# Merge train and val into a single group
s['split_group'] = s['split'].replace({'train': 'train_val',
                                        'val':   'train_val',
                                        'test':  'test'})

for split in ['train_val', 'test']:
    ids = set(s[s['split_group'] == split]['admission_id'].astype(str))
    sub = c[c['admission_id'].astype(str).isin(ids)]
    wins = w[w['admission_id'].astype(str).isin(ids)]

    label = 'Train/Val' if split == 'train_val' else 'Test'
    print(f"\n{label}: {len(sub):,} admissions, "
          f"{sub['subject_id'].nunique():,} patients, "
          f"{len(wins):,} windows")
    print(f"  AKI: {int(sub['aki_detected'].sum())} "
          f"({sub['aki_detected'].mean()*100:.1f}%)")
    print(f"  No AKI: {int((~sub['aki_detected'].astype(bool)).sum())} "
          f"({(~sub['aki_detected'].astype(bool)).mean()*100:.1f}%)")
    print(f"  Sex M: {(sub['sex']=='Male').sum()} "
          f"({(sub['sex']=='Male').mean()*100:.1f}%)")
    print(f"  Sex F: {(sub['sex']=='Female').sum()} "
          f"({(sub['sex']=='Female').mean()*100:.1f}%)")
    print(f"  Age mean±SD: {sub['age_years'].mean():.2f} "
          f"± {sub['age_years'].std():.2f}")
    print(f"  Age median (IQR): {sub['age_years'].median():.1f} "
          f"({sub['age_years'].quantile(0.25):.1f}–"
          f"{sub['age_years'].quantile(0.75):.1f})")
    if 'age_band' in sub.columns:
        print(f"  Age bands:")
        print(sub['age_band'].value_counts().to_string())
    if 'aki_stage' in sub.columns:
        aki_sub = sub[sub['aki_detected'].astype(bool)]
        print(f"  AKI stage (of AKI admissions):")
        print(aki_sub['aki_stage'].value_counts().sort_index().to_string())
    # Temporal label
    if 'onset_dt' in sub.columns or 'hours_to_onset' in wins.columns:
        # Use windows to derive temporal label
        aki_ids = set(sub[sub['aki_detected'].astype(bool)]['admission_id'].astype(str))
        first_onset = (wins[wins['admission_id'].astype(str).isin(aki_ids)]
                       .groupby('admission_id')['hours_to_onset']
                       .min())
        within_3d  = (first_onset <= 72).sum()
        within_7d  = ((first_onset > 72) & (first_onset <= 168)).sum()
        no_onset   = len(sub) - len(aki_ids)
        print(f"  Temporal label:")
        print(f"    Onset ≤3 days:  {within_3d} ({within_3d/len(sub)*100:.1f}%)")
        print(f"    Onset 3–7 days: {within_7d} ({within_7d/len(sub)*100:.1f}%)")
        print(f"    No onset:       {no_onset}  ({no_onset/len(sub)*100:.1f}%)")

In [ ]:
for split in ['train_val', 'test']:
    label = split
    ids = set(s[s['split_group'] == split]['admission_id'].astype(str))
    sub = c[c['admission_id'].astype(str).isin(ids)]
    wins = w[w['admission_id'].astype(str).isin(ids)]
    wins.drop_duplicates(subset='admission_id', keep = 'first', inplace=True)
    print(f"{label} Ethnicity counts and percentages")
    print(f"  white: {int(wins['ethnicity_white'].sum())} "
          f"({wins['ethnicity_white'].mean()*100:.1f}%)")
    print(f"  black: {int(wins['ethnicity_black'].sum())} "
          f"({wins['ethnicity_black'].mean()*100:.1f}%)")
    print(f"  asian: {int(wins['ethnicity_asian'].sum())} "
          f"({wins['ethnicity_asian'].mean()*100:.1f}%)")
    print(f"  mixed: {int(wins['ethnicity_mixed'].sum())} "
          f"({wins['ethnicity_mixed'].mean()*100:.1f}%)")
    print(f"  other: {int(wins['ethnicity_other'].sum())} "
          f"({wins['ethnicity_other'].mean()*100:.1f}%)")
    print(f"  NA: {int(wins['ethnicity_not_available'].sum())} "
          f"({wins['ethnicity_not_available'].mean()*100:.1f}%)")
    